In [1]:
import venti
import numpy as np

In [2]:
print(dir(venti))

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_version', 'models']


In [3]:
from venti.models import plate_motion

In [4]:
# Your GPS data
longitude = np.array([-120.0, -115.0, -110.0, -105.0, -100.0])
latitude = np.array([35.0, 40.0, 45.0, 40.0, 35.0])
velocity_east = np.array([2.1, 1.8, 1.5, 1.2, 0.9])  # mm/year
velocity_north = np.array([0.5, 0.8, 1.1, 1.4, 1.7])  # mm/year
sigma_east = np.array([0.1, 0.15, 0.12, 0.11, 0.13])  # mm/year
sigma_north = np.array([0.12, 0.14, 0.11, 0.13, 0.15])  # mm/year

# Calculate Euler pole
euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, velocity_east, velocity_north,
    sigma_east, sigma_north
)

print(f"Euler Pole: {euler_lon:.2f}°, {euler_lat:.2f}°")
print(f"Angular velocity: {omega:.3f} °/Myr")
print(f"RMS: {stats['rms']:.2f} mm/year")


Euler Pole: -132.10°, 55.03°
Angular velocity: 0.044 °/Myr
RMS: 0.60 mm/year


In [5]:
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
            euler_lon, euler_lat, omega, stats['parameter_covariance'])

In [6]:
# Display results
print(f"\nEULER POLE RESULTS:")
print(f"  Longitude: {euler_lon:.3f}° ± {max_sigma:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}° ± {min_sigma:.3f}°")
print(f"  Rotation:  {omega:.4f} ± {sigma_omega:.4f} °/Myr")

print(f"\nUNCERTAINTY ELLIPSE:")
print(f"  Semi-major axis: {max_sigma:.3f}°")
print(f"  Semi-minor axis: {min_sigma:.3f}°")
print(f"  Ellipse azimuth: {azimuth:.1f}°")
print(f"  Ellipticity: {max_sigma/min_sigma:.2f}")

print(f"\nFIT QUALITY STATISTICS:")
print(f"  RMS:              {stats['rms']:.2f} mm/year")
print(f"  Weighted RMS:     {stats['wrms']:.2f}")
print(f"  Chi-squared:      {stats['chi_squared']:.2f}")
print(f"  Reduced chi-sq:   {stats['reduced_chi_squared']:.2f}")
print(f"  Degrees of freedom: {stats['degrees_of_freedom']}")

print(f"\nINTERPRETATION:")
if stats['reduced_chi_squared'] < 1.5:
    print(f"  Good fit: reduced χ² = {stats['reduced_chi_squared']:.2f} < 1.5")
elif stats['reduced_chi_squared'] < 3.0:
    print(f"  Acceptable fit: reduced χ² = {stats['reduced_chi_squared']:.2f}")
else:
    print(f"  Poor fit: reduced χ² = {stats['reduced_chi_squared']:.2f} > 3.0")

if max_sigma < 5.0:
    print(f"  Good position uncertainty: {max_sigma:.2f}° < 5°")
elif max_sigma < 10.0:
    print(f"  Moderate position uncertainty: {max_sigma:.2f}°")
else:
    print(f"  Large position uncertainty: {max_sigma:.2f}° > 10°")

if sigma_omega / omega < 0.1:
    print(f"  Good rotation rate uncertainty: {sigma_omega/omega:.1%} < 10%")
elif sigma_omega / omega < 0.2:
    print(f"  Moderate rotation rate uncertainty: {sigma_omega/omega:.1%}")
else:
    print(f"  Large rotation rate uncertainty: {sigma_omega/omega:.1%} > 20%")


EULER POLE RESULTS:
  Longitude: -132.095° ± 2.148°
  Latitude:  55.032° ± 0.605°
  Rotation:  0.0439 ± 0.0040 °/Myr

UNCERTAINTY ELLIPSE:
  Semi-major axis: 2.148°
  Semi-minor axis: 0.605°
  Ellipse azimuth: -39.3°
  Ellipticity: 3.55

FIT QUALITY STATISTICS:
  RMS:              0.60 mm/year
  Weighted RMS:     0.41
  Chi-squared:      122.52
  Reduced chi-sq:   4.18
  Degrees of freedom: 7

INTERPRETATION:
  Poor fit: reduced χ² = 4.18 > 3.0
  Good position uncertainty: 2.15° < 5°
  Good rotation rate uncertainty: 9.0% < 10%


In [9]:
modeled_vel = plate_motion.model_velocities_from_euler_pole(longitude, latitude, -107.2, 50.8, 0.65)
modeled_vel

array([[ 0.02054729, -0.01012073],
       [ 0.01404486, -0.00619798],
       [ 0.00757972, -0.00223026],
       [ 0.01379588,  0.00175313],
       [ 0.02010477,  0.00572545]])

In [10]:
modeled_vel, uncertainties = plate_motion.model_velocities_from_euler_pole(
    longitude, latitude, euler_lon, euler_lat, omega,
    euler_covariance=stats['parameter_covariance']  # From calculate_euler_pole()
)
modeled_vel, uncertainties

(array([[0.00172018, 0.00058557],
        [0.00135892, 0.00082127],
        [0.0010096 , 0.00105058],
        [0.0014762 , 0.00127251],
        [0.00192863, 0.00148486]]),
 array([[6.28215290e-05, 8.22999796e-05],
        [5.32382402e-05, 6.34788657e-05],
        [7.26954914e-05, 5.74673951e-05],
        [5.42226280e-05, 6.79417191e-05],
        [6.50193022e-05, 8.90457201e-05]]))

In [25]:
itrfplates = plate_motion.json_to_dataframe(plate_motion.load_itrf14_json())
itrfplates

,plate,name,omega_x,omega_y,omega_z
0,ANTA,Antarctic Plate,-0.0688,-0.0900,0.1874
1,ARAB,Arabian Plate,0.3205,-0.0378,0.4011
2,AUST,Australian Plate,0.4194,0.3284,0.3375
3,EURA,Eurasian Plate,-0.0235,-0.1476,0.2140
4,INDI,Indian Plate,0.3205,-0.0014,0.4038
5,NAZC,Nazca Plate,-0.0925,-0.4290,0.4508
6,NOAM,North American Plate,0.0066,-0.1928,-0.0176
7,NUBI,Nubian Plate,0.0274,-0.1704,0.2037
8,PCFC,Pacific Plate,-0.1135,0.2907,-0.6025
9,SOAM,South American Plate,-0.0751,-0.0835,-0.0389


In [19]:
ITRF_GPS_DATA = """CODE DOMES #   Site Name        Plate       T   Lon     Lat       Ve     Vn     Se     Sn    Re     Rn
MAW1 66004M001 Mawson station   Antarctica  P     62.871 -67.605  -3.73  -2.06  0.006  0.006  0.272 -0.190
DAV1 66010M001 Davis            Antarctica  P     77.973 -68.577  -2.93  -5.14  0.007  0.007  0.009 -0.250
WHN0 66069M001 Westhaven nunata Antarctica  P    154.220 -79.846   5.60 -12.25  0.032  0.036  0.589 -0.063
FLM5 66061M002 Mount fleming    Antarctica  P    160.271 -77.533   8.15 -12.07  0.020  0.023  0.101  0.087
FTP4 66062M002 Fishtail point   Antarctica  P    162.565 -78.928   8.18 -11.74  0.011  0.012  0.050 -0.075
ROB4 66063M002 Cape roberts     Antarctica  P    163.190 -77.034   9.01 -11.79  0.008  0.009 -0.004  0.023
MCM4 66001M003 Mac murdo        Antarctica  P    166.669 -77.838   9.68 -11.48  0.007  0.007 -0.260  0.003
HALY 20102M001 Halat ammar      Arabia      P     36.100  29.139  27.01  23.80  0.012  0.011 -0.727  0.601
NAMA 20103M001 Namas            Arabia      P     42.045  19.211  34.53  27.57  0.008  0.007 -0.078 -0.553
JIZN 20131M001 Jizan            Arabia      P     42.104  16.699  35.98  26.98  0.041  0.035  0.051  0.049
SOLA 20101M001 Solar village -  Arabia      LP    46.401  24.911  31.72  29.33  0.007  0.007 -0.218 -0.611
BAHR 24901M002 Bahrein (juffar) Arabia      P     50.608  26.209  31.18  30.28  0.005  0.005  0.394 -0.059
YAR1 50107M004 Yarragadee       Australia   DLPR 115.347 -29.047  38.85  57.88  0.007  0.006  0.324 -0.083
PERT 50133M001 Perth            Australia   PR   115.885 -31.802  38.96  57.95  0.009  0.008 -0.443 -0.047
NNOR 50181M001 New norcia       Australia   P    116.193 -31.049  38.41  58.13  0.007  0.006  0.057 -0.152
KARR 50139M001 Karratha         Australia   P    117.097 -20.981  38.88  58.41  0.006  0.005  0.228 -0.218
ALBY 50191M001 Albany           Australia   P    117.810 -34.950  37.14  58.13  0.018  0.017 -0.296  0.159
ESPA 50177M002 Esperance        Australia   P    121.894 -33.874  34.84  58.91  0.016  0.016 -0.081 -0.016
BRO1 50176M003 Broome           Australia   P    122.209 -18.004  37.31  59.62  0.034  0.029  0.291 -0.654
DARM 50184M001 Darwin arms      Australia   P    130.891 -12.424  35.84  59.18  0.014  0.012  0.225  0.035
DARW 50134M001 Darwin i         Australia   P    131.133 -12.844  35.74  59.40  0.007  0.006  0.193 -0.197
KAT1 59968M001 Katherine - nort Australia   PR   132.153 -14.376  35.05  59.25  0.016  0.014  0.303 -0.112
JAB1 50136M001 Jabiru           Australia   P    132.894 -12.659  35.38  58.67  0.024  0.021  0.195  0.405
CEDU 50138M001 Ceduna           Australia   P    133.810 -31.867  28.97  58.75  0.006  0.005 -0.153  0.194
ALIC 50137M001 Alice springs -  Australia   P    133.886 -23.670  32.01  59.16  0.006  0.005  0.020 -0.193
SA45 59987M001 Saltier          Australia   P    137.934 -32.470  26.37  58.14  0.027  0.027 -0.065  0.222
ADE1 50109S001 Salisbury        Australia   P    138.647 -34.729  24.65  58.45  0.007  0.006  0.093 -0.229
PTLD 50158M003 Portland, victor Australia   P    141.613 -38.344  20.95  57.11  0.039  0.044 -0.018  0.459
HNIS 59959M001 Horn island - qu Australia   P    142.296 -10.590  34.38  57.35  0.058  0.052 -0.106  0.123
MOBS 50182M001 Melbourne observ Australia   P    144.975 -37.829  19.28  57.02  0.007  0.006 -0.092 -0.363
STNY 50160M002 Stony point - vi Australia   P    145.214 -38.375  18.97  56.35  0.052  0.058 -0.272  0.232
BUR2 50144M003 Burnie           Australia   P    145.915 -41.050  16.90  56.41  0.018  0.020 -0.404 -0.046
TOW2 50140M001 Townsville - cap Australia   P    147.056 -19.269  28.87  55.97  0.006  0.005  0.279  0.077
SPBY 50162M004 Spring bay - hob Australia   P    147.931 -42.546  13.80  56.11  0.027  0.033  0.369 -0.424
PARK 50108M001 Parkes           Australia   PR   148.265 -32.999  20.11  54.99  0.013  0.013  0.338  0.611
TIDB 50103M108 Tidbinbilla      Australia   DLPR 148.980 -35.399  18.18  55.45  0.007  0.006  0.282 -0.114
STR1 50119M002 Mount stromlo    Australia   DLP  149.010 -35.316  18.31  55.40  0.007  0.007  0.200 -0.074
RSBY 59953M001 Roslyn bay       Australia   P    150.790 -23.161  25.80  54.60  0.091  0.083 -0.197  0.067
PTKL 50145M004 Auckland ii      Australia   P    150.914 -34.476  18.17  54.66  0.036  0.039 -0.131 -0.076
SYDN 50124M003 Sydney           Australia   P    151.150 -33.781  18.14  54.47  0.007  0.007  0.261  0.018
BNDY 50185M001 Bundaberg argn   Australia   P    152.321 -24.908  23.96  54.39  0.013  0.012 -0.058 -0.365
SUNM 50143M001 Brisbane - wooll Australia   P    153.035 -27.485  21.71  53.83  0.010  0.009  0.158 -0.114
CLEV 59978M001 Cleveland        Australia   P    153.267 -27.526  22.01  53.81  0.039  0.040 -0.272 -0.191
LORD 59998M001 Lord howe island Australia   P    159.061 -31.520  16.09  50.89  0.020  0.021  0.094 -0.106
NMEA 92733M001 Noumea - dittt   Australia   D    166.443 -22.265  20.10  47.42  0.105  0.096  0.823 -0.956
NRMD 92701M005 Noumea           Australia   DP   166.485 -22.228  20.41  46.52  0.012  0.011  0.533 -0.083
NORF 50189M001 Norfolk island   Australia   P    167.939 -29.043  14.04  45.46  0.017  0.016  0.480  0.015
TAKL 50216S001 Auckland tide ga Australia   P    174.770 -36.844   4.32  40.16  0.016  0.017  0.041  0.454
HERS 13212M007 Herstmonceux cas Eurasia     LP     0.336  50.867  16.80  16.57  0.005  0.006  0.356 -0.191
EBRE 13410M001 Roquetes - torto Eurasia     P      0.492  40.821  19.97  16.23  0.006  0.006 -0.130  0.146
SHOE 19197M001 Shoeburyness     Eurasia     P      0.827  51.555  17.52  16.42  0.016  0.020 -0.457 -0.068
ESCO 13435M001 Escornacrabes    Eurasia     P      0.976  42.694  19.31  16.52  0.007  0.007  0.184 -0.161
AMBL 19967M001 Ambrumesnil      Eurasia     P      0.994  49.859  17.45  15.94  0.020  0.025  0.139  0.404
WEYB 19178M001 Weybourne        Eurasia     P      1.130  52.945  16.76  16.20  0.018  0.023 -0.046  0.138
VIGO 13450M001 Vigo             Eurasia     P    351.187  42.184  17.64  16.98  0.010  0.011  0.082 -0.369
GAIA 13902M001 Porto            Eurasia     P    351.411  41.106  17.70  17.01  0.011  0.014  0.358 -0.403
NEWL 13273M103 Bartinney        Eurasia     P    354.457  50.103  15.81  16.73  0.006  0.007  0.270 -0.166
HOLY 19146M001 Holyhead         Eurasia     P    355.358  53.318  15.58  16.61  0.014  0.017 -0.301 -0.059
BRST 10004M004 Brest            Eurasia     P    355.503  48.380  16.70  16.93  0.008  0.009  0.134 -0.382
MADR 13407S012 Madrid-robledo   Eurasia     PR   355.750  40.429  18.82  16.43  0.007  0.007  0.236  0.127
YEBE 13420M001 Yebes            Eurasia     PR   356.911  40.525  18.89  16.41  0.004  0.004  0.357  0.109
WTZR 14201M010 Wettzell         Eurasia     LPR   12.879  49.144  20.33  15.57  0.004  0.004 -0.044 -0.170
POTS 14106M003 Potsdam          Eurasia     LP    13.066  52.379  19.22  15.23  0.005  0.004  0.283  0.145
MALD 22901S001 Male airport     India       P     73.526   4.189  43.95  34.56  0.028  0.022  0.154 -0.302
HYDE 22307M001 Hyderabad        India       P     78.551  17.417  40.63  35.18  0.012  0.010  0.210 -0.182
LCKI 22305M002 Lucknow          India       P     80.956  26.912  37.20  35.07  0.102  0.094  0.455  0.167
NISU 49507M001 Boulder, colorad N. America  P    254.738  39.995 -14.64  -5.92  0.015  0.016  0.017 -0.437
AMC2 40472S004 Colorado springs N. America  P    255.475  38.803 -14.49  -5.71  0.006  0.006  0.125 -0.380
GODE 40451M123 Washington       N. America  DPLR 283.173  39.022 -14.69   4.16  0.004  0.003 -0.028  0.005
BRMU 42501S004 Bermuda          N. America  P    295.304  32.370 -11.98   8.90  0.005  0.004 -0.171 -0.398
EISL 41703M003 Easter island    Nazca       LP   250.617 -27.148  66.73  -6.07  0.007  0.007 -0.070 -0.064
GALA 42005M001 Santa cruz       Nazca       P    269.696  -0.743  50.98  10.32  0.008  0.007 -0.189 -0.279
WIND 31101M001 Windhoek         Nubia       P     17.089 -22.575  19.41  19.68  0.010  0.009  0.533 -0.652
SUTH 30314M002 Sutherland       Nubia       P     20.810 -32.380  16.90  19.25  0.006  0.006  0.205 -0.441
MAS1 31303M002 Maspalomas       Nubia       P    344.367  27.764  16.58  17.48  0.005  0.005 -0.230 -0.045
POHN 51601M001 Pohnpei          Pacific     P    158.210   6.960 -70.04  25.43  0.012  0.010  0.606 -0.065
KWJ1 50506M001 Kwajalein atoll  Pacific     PR   167.730   8.722 -69.05  29.60  0.020  0.016 -0.140 -0.659
KIRI 50305M001 Betio island - k Pacific     P    172.923   1.355 -67.83  31.17  0.010  0.008  0.389 -0.611
KOKB 40424M004 Kauai            Pacific     DPR  200.335  22.126 -62.21  34.62  0.007  0.006 -0.185  0.110
HNLC 49970S001 Honolulu         Pacific     P    202.135  21.303 -62.70  34.68  0.006  0.005  0.364  0.050
MKEA 40477M001 Mauna kea        Pacific     PR   204.544  19.801 -62.64  35.03  0.006  0.006  0.136 -0.335
THTI 92201M009 Papeete (tahiti) Pacific     DP   210.394 -17.577 -65.81  34.34  0.007  0.006  0.208 -0.041
IQUI 42204M001 Iquitos          S. America  P    286.731  -3.767  -3.98  10.64  0.019  0.015  0.089  0.041
RIOB 41645M001 Rio branco       S. America  P    292.197  -9.965  -2.79  11.28  0.020  0.016 -0.533 -0.036
KOUR 97301M210 Kourou           S. America  DP   307.194   5.252  -5.10  13.01  0.009  0.008  0.575 -0.734
BRAZ 41606M001 Brasilia         S. America  P    312.122 -15.947  -3.90  12.52  0.006  0.006  0.088 -0.087
ASC1 30602M001 Ascension iii    S. America  P    345.588  -7.951  -5.39  11.11  0.008  0.007  0.305 -0.029
MALI 33201M001 Malindi          Somalia     P     40.194  -2.996  26.17  16.56  0.008  0.007  0.145 -0.212
SEY1 39801M001 Mahe island      Somalia     P     55.479  -4.674  25.00  11.06  0.015  0.013  0.425 -0.226"""

import pandas as pd
def parse_itrf_gps_data():
    """
    Parse the ITRF2014 GPS velocity data into a pandas DataFrame.
    """
    # Read the data, skipping the header
    lines = ITRF_GPS_DATA.strip().split('\n')[1:]  # Skip header
    
    data = []
    for line in lines:
        # Parse each line - fixed width format
        if len(line) < 80:  # Skip short lines
            continue
            
        parts = line.split()
        if len(parts) < 12:  # Skip incomplete lines
            continue
            
        try:
            code = parts[0]
            # Skip DOMES and Site Name (variable length)
            # Find plate name and technique
            plate_idx = None
            for i, part in enumerate(parts):
                if part in ['Antarctica', 'Arabia', 'Australia', 'Eurasia', 'India', 
                           'N.', 'Nazca', 'Nubia', 'Pacific', 'S.', 'Somalia']:
                    plate_idx = i
                    break
            
            if plate_idx is None:
                continue
                
            # Handle "N. America" and "S. America"
            if parts[plate_idx] == 'N.':
                plate = 'N. America'
                tech_idx = plate_idx + 2
            elif parts[plate_idx] == 'S.':
                plate = 'S. America'  
                tech_idx = plate_idx + 2
            else:
                plate = parts[plate_idx]
                tech_idx = plate_idx + 1
            
            technique = parts[tech_idx]
            
            # Extract numerical values (last 8 columns)
            numerical_parts = parts[-8:]
            lon, lat, ve, vn, se, sn, re, rn = map(float, numerical_parts)
            
            data.append({
                'CODE': code,
                'Plate': plate,
                'Technique': technique,
                'Longitude': lon,
                'Latitude': lat,
                'Ve': ve,
                'Vn': vn,
                'Se': se,
                'Sn': sn,
                'Re': re,
                'Rn': rn
            })
            
        except (ValueError, IndexError) as e:
            print(f"Warning: Could not parse line: {line[:50]}...")
            continue
    
    return pd.DataFrame(data)

In [21]:
gps_data = parse_itrf_gps_data()
gps_data

,CODE,Plate,Technique,Longitude,Latitude,Ve,Vn,Se,Sn,Re,Rn
0,MAW1,Antarctica,P,62.871,-67.605,-3.73,-2.06,0.006,0.006,0.272,-0.190
1,DAV1,Antarctica,P,77.973,-68.577,-2.93,-5.14,0.007,0.007,0.009,-0.250
2,WHN0,Antarctica,P,154.220,-79.846,5.60,-12.25,0.032,0.036,0.589,-0.063
3,FLM5,Antarctica,P,160.271,-77.533,8.15,-12.07,0.020,0.023,0.101,0.087
4,FTP4,Antarctica,P,162.565,-78.928,8.18,-11.74,0.011,0.012,0.050,-0.075
...,...,...,...,...,...,...,...,...,...,...,...
84,KOUR,S. America,DP,307.194,5.252,-5.10,13.01,0.009,0.008,0.575,-0.734
85,BRAZ,S. America,P,312.122,-15.947,-3.90,12.52,0.006,0.006,0.088,-0.087
86,ASC1,S. America,P,345.588,-7.951,-5.39,11.11,0.008,0.007,0.305,-0.029
87,MALI,Somalia,P,40.194,-2.996,26.17,16.56,0.008,0.007,0.145,-0.212


In [27]:
pacific_data = gps_data[gps_data['Plate'] == 'Pacific']

# Calculate Euler pole
longitude = pacific_data['Longitude'].values
latitude = pacific_data['Latitude'].values
ve = pacific_data['Ve'].values
vn = pacific_data['Vn'].values
se = pacific_data['Se'].values
sn = pacific_data['Sn'].values

euler_lon, euler_lat, omega, stats = plate_motion.calculate_euler_pole(
    longitude, latitude, ve, vn, se, sn
)

# Calculate uncertainties
max_sigma, min_sigma, azimuth, sigma_omega = plate_motion.get_euler_pole_uncertainty(
    euler_lon, euler_lat, omega, stats['parameter_covariance']
)

print(f"\nCalculated Pacific Plate Euler Pole:")
print(f"  Longitude: {euler_lon:.3f}°")
print(f"  Latitude:  {euler_lat:.3f}°")
print(f"  Angular velocity: {omega:.4f} °/Myr")
print(f"  RMS: {stats['rms']:.2f} mm/year")


Calculated Pacific Plate Euler Pole:
  Longitude: 110.740°
  Latitude:  -62.643°
  Angular velocity: 0.6812 °/Myr
  RMS: 0.36 mm/year


In [34]:
plate_motion.euler_pole_to_rotation_rate(euler_lon, euler_lat, np.rad2deg(omega)*1e6)

(np.float64(-0.1108594649290671),
 np.float64(0.2927566175915871),
 np.float64(-0.6050402382540072))

In [26]:
itrfplates[itrfplates.name == 'Pacific Plate'] 

,plate,name,omega_x,omega_y,omega_z
8,PCFC,Pacific Plate,-0.1135,0.2907,-0.6025
